In [ ]:
import sys; sys.path.append('..')
import MeshFEM, mesh, mesh_energy, param_utils, viewer, benchmark
import numpy as np
import sim_utils

import matplotlib
from matplotlib import pyplot as plt
import visualization

import newton_flow

In [ ]:
# m_rest = mesh.Mesh('data/pants_rest.obj')
# m_defo = mesh.Mesh('data/pants_deformed.obj')
m_rest = mesh.Mesh('data/Hilbert_opt_2d.obj')
m_defo = mesh.Mesh('data/Hilbert_init_2d.obj')
# m_defo.setVertices(m_defo.vertices() / np.sqrt(m_defo.volume))
v = mesh_energy.NodalVars(m_rest, 2)
v.setVars(m_defo.vertices().ravel())

In [ ]:
nf = newton_flow.symmetric_dirichlet(m_rest, v)
# nf = newton_flow.linear_elastic(m_rest, v)

In [ ]:
nf.projectionSmoothingEpsilon = 0 # 1e-4 # 1e-8

In [ ]:
import py_newton_optimizer
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(v, [nf])

In [ ]:
# Nullspace pinning strategy
FIX_VARS = False
if FIX_VARS:
    # fv = sim_utils.getBBoxVars(m_rest, sim_utils.BBoxFace.MIN_X)
    # prob.setFixedVars(fv)
    import elastic_solid, energy
    es = elastic_solid.ElasticSolid(m_rest, energy.CommonNeoHookeanYoungPoisson(2, 1, 0.3))
    es.setDeformedPositions(m_defo.vertices())
    pin_vars, _ = es.prepareRigidMotionPins()
    v.setVars(es.getVars())
    prob.setFixedVars(pin_vars)
else:
    # prob.hessianShift = 1e-5
    # nf.elementHessianShift = 1e-8
    # nf.elementHessianShift = 1e-8
    prob.hessianShift = 1e-10
    prob.useRelativeHessianShift = False

In [ ]:
import newton_flow_utils

In [ ]:
constant_speed = True
always_project = True

In [ ]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
import visualization
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

In [ ]:
import newton_flow_utils as nfu
fv = nfu.ground_truth_flow(opt, 1.0, verbose=True, grad_tol=1e-8)

In [ ]:
# Experiment with ways of detecting onset of "convergence plateau"
def moving_average(x, w):
    return np.convolve(x, np.ones(w), 'valid') / w

g =  visualization.trajectory_gnorms(prob, fv)
ratios = np.clip(g[:-1] / g[1:], 0, 1.5)
plt.plot(ratios)
plt.plot(moving_average(ratios, 5))

In [ ]:
opt.options.niter = 0
opt.optimize()
opt.update_factorizations()

In [ ]:
always_project = False
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
import importlib
importlib.reload(visualization);
importlib.reload(nfu);

extrapolation_dist = 10
constant_speed = True
# constant_speed = False
num_frames = min(500, len(fv))

methods = [
    (1, nfu.eval_trajectory_taylor, 'Newton'),
    # (2, nfu.eval_trajectory_logspiral, 'Deg 2 spiral'),
    (2, nfu.eval_trajectory_taylor, 'Deg 2 Taylor'),
    (3, nfu.eval_trajectory_taylor, 'Deg 3 Taylor'),
    # (4, nfu.eval_trajectory_taylor, 'Deg 4 Taylor'),
    (14, nfu.eval_trajectory_vector_pade, 'Pade 14'),
    (19, nfu.eval_trajectory_vector_pade, 'Pade 19'),
]
ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed, corners_only=True,
                         extrapolation_method_list=methods, truncate=True)

In [ ]:
ff(50)

In [ ]:
visualization.writeVideo('pade_compare.mp4', num_frames, ff)

In [ ]:
extrapolation_dist = 3
constant_speed = True
num_frames = min(500, len(fv))

baseline_methods = [(1, nfu.eval_trajectory_taylor, 'Newton'),
           (2, nfu.eval_trajectory_taylor, 'Deg 2 Taylor'),
           (3, nfu.eval_trajectory_taylor, 'Deg 3 Taylor')]
ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed, corners_only=True,
                         extrapolation_method_list=baseline_methods
                         + [(2, nfu.eval_trajectory_logspiral, 'Deg 2 Spiral')])

visualization.writeVideo('spiral_deg_2_compare.mp4', num_frames, ff)

In [ ]:
ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=baseline_methods
                         + [(3, nfu.eval_trajectory_logspiral, 'Deg 3 Spiral')])
visualization.writeVideo('spiral_deg_3_compare.mp4', num_frames, ff)

# TODO
- Postprocess Newton step to remove rigid motion (does this make the steps more coherent?)
- Try KKT formulation for pinning rigid motion
- See relative performance of symmetric indefinite factorization within Accelerate
- Consider sparse + epsilon * low rank factorization idea for pinning rigid motion (omitting epsilon^2 fully dense term)

In [ ]:
# Corner detection
m = mesh.Mesh('data/Hilbert_opt_2d.obj')

In [ ]:
v = viewer.Viewer(m)
v.show()

In [ ]:
import igl
incidentAngle = np.zeros(m.numVertices())
np.add.at(incidentAngle, m.elements(), igl.internal_angles(m.vertices(), m.elements()))
is_corner = np.logical_and(np.abs(incidentAngle - 2 * np.pi) > 0.01, np.abs(incidentAngle - np.pi) > 0.5)

In [ ]:
v.update(scalarField={'data': is_corner, 'colormap': matplotlib.cm.inferno})

In [ ]:
visualization.plot_mesh(prob.getVars().reshape(-1, 2), m_rest.elements())

# Finite Difference Validation

In [ ]:
proj = False
nc = 16
eps = 1e-4
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways() if proj else py_newton_optimizer.HessianProjectionNever()

In [ ]:
prob.setVars(fv[50].ravel())

In [ ]:
opt.update_factorizations()
d = opt.newton_step()
d_coeffs = nf.computeTaylorCoefficients(opt.hessian_factorization, 16, proj)

In [ ]:
import benchmark, math
prob.disableCaching = True
benchmark.reset()
x = prob.getVars()
prob.setVars(x + eps * d)
d_plus = opt.newton_step()
d_coeffs_plus = nf.computeTaylorCoefficients(opt.hessian_factorization, 16, proj)

prob.setVars(x - eps * d)
d_minus = opt.newton_step()
d_coeffs_minus = nf.computeTaylorCoefficients(opt.hessian_factorization, 16, proj)
d_prime_ad = (d_plus - d_minus) / (2 * eps)
# Note: this finite difference approximation is wrong! we must incorporate the effect of `d_prime_ad` when differencing!
d_pprime_ad = (d_plus + d_minus - 2 * d) / (eps * eps) 

prob.setVars(x + eps * d + 0.5 * (eps * eps) * d_prime_ad)
d_pp = opt.newton_step()
prob.setVars(x - eps * d + 0.5 * (eps * eps) * d_prime_ad)
d_mm = opt.newton_step()
d_pprime_ad2 = (d_pp + d_mm - 2 * d) / (eps * eps)
prob.setVars(x)
# benchmark.report()

In [ ]:
for i in range(nc - 1):
    print(((d_coeffs_plus[i][:5] - d_coeffs_minus[i][:5]) / (2 * eps)) / (d_coeffs[i + 1][:5] * math.factorial(i + 2) / math.factorial(i + 1)))

In [ ]:
opt.update_factorizations()
speed_factor = 2
d_coeffs = [x_i * speed_factor**(i + 1) for i, x_i in enumerate(nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, 16, proj))]

In [ ]:
import benchmark
prob.disableCaching = True
benchmark.reset()
prob.setVars(x + eps * d_coeffs[0])
opt.update_factorizations()
d_coeffs_plus = [x_i * speed_factor**(i + 1) for i, x_i in enumerate(nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, 16, proj))]

prob.setVars(x - eps * d_coeffs[0])
opt.update_factorizations()
d_coeffs_minus = [x_i * speed_factor**(i + 1) for i, x_i in enumerate(nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, 16, proj))]
prob.setVars(x)

In [ ]:
for i in range(nc - 1):
    print(((d_coeffs_plus[i][:5] - d_coeffs_minus[i][:5]) / (2 * eps)) / (d_coeffs[i + 1][:5] * math.factorial(i + 2) / math.factorial(i + 1)))